# Day 15 · Inside out — From the End of the Line

**Question.** The address grid promises equal distance: 800 numbers to the mile in every direction.
How far is the rest of Chicago from 130th Street, where the Red Line Extension will end, when the
distance is counted in minutes by bus and train?

**Approach.** Door-to-door transit minutes (CTA, Metra and Pace schedules, a Tuesday morning) from two
starts, 130th Street and State & Madison, to every quarter-mile point of the address grid. The main map
shades the city by the minutes from 130th Street. The middle column turns the map inside out: each point
keeps its direction from the start and sits as far out as its minutes, so the minutes become circles and
the square-mile grid bends.

| Section | What happens |
| --- | --- |
| 1. Setup | Imports, paths, settings |
| 2. Pull | GTFS feeds on disk (CTA, Metra, Pace), city limits, 2020 Census blocks |
| 3. Validate | Service day, trips, stops |
| 4. Transform | Connections for the morning; the quarter-mile lattice |
| 5. Analyze | Earliest arrival by connection scan; minutes per point; residents |
| 6. QA | Another hour; walking points |
| 7. Export | `plate.json` |

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import logging
import zipfile
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter
from scipy.spatial import cKDTree
from shapely import contains_xy
from shapely.geometry import Polygon

from atlas.grid import miles_from_origin
from atlas.ledger import Ledger, write_plate

logging.basicConfig(level=logging.INFO, format='%(message)s')
log = logging.getLogger('day15')
DAY = Path('.')
REPO = DAY.resolve().parents[1]
RAW = DAY / 'data' / 'raw'
PROC = DAY / 'data' / 'processed'
OUT = DAY / 'output'
for p in (RAW, PROC, OUT):
    p.mkdir(parents=True, exist_ok=True)
GTFS = REPO / 'reference' / 'raw' / 'gtfs'

SERVICE_DAY = date(2026, 11, 17)          # an ordinary Tuesday
DEPARTS = list(range(8 * 3600, 9 * 3600, 300))  # leave every 5 minutes, 8:00-8:55 a.m.
HORIZON = 210 * 60                        # stop scanning 3.5 hours after leaving
WALK_MPS, DETOUR = 1.2, 1.3               # walking speed; straight line x 1.3 for the streets
WALK = WALK_MPS / DETOUR / 1609.344       # straight-line miles per second
ACCESS, XFER, XFER_S = 0.75, 0.25, 60     # miles to a stop; miles between stops (+ seconds)
STEP = 0.25                               # lattice, miles
WATER = 180                               # minutes: no stop within reach (Lake Calumet, the lake)
FEEDS = {
    'cta': ('c', 'CTA', 'https://www.transitchicago.com/developers/gtfs/'),
    'metra': ('m', 'Metra', 'https://metra.com/developers'),
    'pace': ('p', 'Pace', 'https://www.pacebus.com/route-timetable-data-services'),
}
# 130th Street on the Red Line Extension alignment (OpenStreetMap railway=construction, saved for Day 08):
# the extension's last station, opening 2030. State & Madison is the grid's origin.
RLE = REPO / 'days' / 'day08-utopia' / 'data' / 'raw' / 'osm_rle_2026-11-01.geojson'
HOSPITAL = (41.8954, -87.6206)            # Northwestern Memorial Hospital, 251 E Huron St

## 2. Pull
Everything is on disk: the GTFS snapshots used since Day 09, the city limits and the Census blocks.

In [ ]:
def latest(prefix):
    return sorted(GTFS.glob(f'{prefix}_gtfs_*.zip'))[-1]

feeds = {k: latest(k) for k in FEEDS}
for k, z in feeds.items():
    log.info('%s %s', k, z.name)
city = json.load(open(REPO / 'reference' / 'city-boundary.json'))
blocks = pd.read_csv(REPO / 'reference' / 'chicago-blocks-2020.csv', dtype={'geoid': str})
rle = json.load(open(RLE))
log.info('blocks %d, residents %d', len(blocks), blocks['pop'].sum())

## 3. Validate

In [ ]:
def rd(z, name, **kw):
    """Read a GTFS table; Metra pads its fields with spaces."""
    df = pd.read_csv(z.open(name), dtype=str, skipinitialspace=True, **kw)
    df.columns = df.columns.str.strip()
    return df.apply(lambda c: c.str.strip())


def service_ids(z, day):
    cal = rd(z, 'calendar.txt')
    stamp, wd = day.strftime('%Y%m%d'), day.strftime('%A').lower()
    ids = set(cal[(cal['start_date'] <= stamp) & (cal['end_date'] >= stamp) & (cal[wd] == '1')]['service_id'])
    exc = rd(z, 'calendar_dates.txt')
    exc = exc[exc['date'] == stamp]
    ids |= set(exc.loc[exc['exception_type'] == '1', 'service_id'])
    ids -= set(exc.loc[exc['exception_type'] == '2', 'service_id'])
    return ids


trips, stops = {}, []
for k, path in feeds.items():
    z = zipfile.ZipFile(path)
    t = rd(z, 'trips.txt', usecols=['route_id', 'service_id', 'trip_id'])
    t = t[t['service_id'].isin(service_ids(z, SERVICE_DAY))]
    trips[k] = t
    s = rd(z, 'stops.txt', usecols=['stop_id', 'stop_lat', 'stop_lon'])
    s['stop_id'] = FEEDS[k][0] + s['stop_id']
    stops.append(s)
    log.info('%s: %d trips on %s, %d routes', k, len(t), SERVICE_DAY, t['route_id'].nunique())
    assert len(t) > 500, f'{k}: no service on {SERVICE_DAY}?'
stops = pd.concat(stops).drop_duplicates('stop_id').reset_index(drop=True)

## 4. Transform
Every hop between consecutive stops of a trip leaving between 7 and 11 a.m., sorted by departure.

In [ ]:
def secs(s):
    p = s.str.split(':', expand=True).astype(int)
    return p[0] * 3600 + p[1] * 60 + p[2]


cache = PROC / 'connections.npz'
idx = {s: i for i, s in enumerate(stops['stop_id'])}
if cache.exists():
    conn = np.load(cache)['conn']
else:
    parts = []
    for k, path in feeds.items():
        z, pre = zipfile.ZipFile(path), FEEDS[k][0]
        want = set(trips[k]['trip_id'])
        rows = []
        for ch in pd.read_csv(z.open('stop_times.txt'), dtype=str, skipinitialspace=True, chunksize=2_000_000,
                              usecols=lambda c: c.strip() in {'trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence'}):
            ch.columns = ch.columns.str.strip()
            ch = ch.apply(lambda c: c.str.strip())
            ch = ch[ch['trip_id'].isin(want)].dropna(subset=['departure_time'])
            ch = ch.assign(dep=secs(ch['departure_time']), arr=secs(ch['arrival_time']))
            rows.append(ch[(ch['dep'] >= 7 * 3600) & (ch['dep'] <= 11 * 3600)])
        r = pd.concat(rows)
        r['seq'] = r['stop_sequence'].astype(int)
        r = r.sort_values(['trip_id', 'seq'])
        nxt = r.shift(-1)
        ok = (nxt['trip_id'] == r['trip_id']).to_numpy()
        parts.append(pd.DataFrame({'trip': (pre + r['trip_id'])[ok], 'a': (pre + r['stop_id'])[ok],
                                   'b': (pre + nxt['stop_id'])[ok], 'dep': r['dep'][ok], 'arr': nxt['arr'][ok]}))
    c = pd.concat(parts)
    c = c[c['a'].isin(idx) & c['b'].isin(idx)]
    tid = {t: i for i, t in enumerate(c['trip'].unique())}
    conn = np.column_stack([c['dep'], c['arr'], c['a'].map(idx), c['b'].map(idx), c['trip'].map(tid)]).astype(np.int64)
    conn = conn[np.argsort(conn[:, 0], kind='stable')]
    np.savez(cache, conn=conn)
assert (conn[:, 1] >= conn[:, 0]).all(), 'a hop arrives before it leaves'
sx, sy = miles_from_origin(stops['stop_lat'].astype(float), stops['stop_lon'].astype(float))
pts = np.column_stack([sx, sy])
tree = cKDTree(pts)
fp = [[] for _ in range(len(pts))]
for i, j in tree.query_pairs(XFER):
    w = np.hypot(*(pts[i] - pts[j])) / WALK + XFER_S
    fp[i].append((j, w))
    fp[j].append((i, w))
log.info('%d hops, %d stops, %d trips', len(conn), len(pts), conn[:, 4].max() + 1)

In [ ]:
# The lattice: every quarter mile of the address grid, in miles from State & Madison (the plates' miles:
# 800 numbers a mile, 1,200 for the first mile south).
def to_miles(a):
    a = np.asarray(a, float)
    x, y = a[:, 0] / 800, a[:, 1]
    y = np.where(y >= 0, y / 800, np.where(y >= -1200, y / 1200, -1 + (y + 1200) / 800))
    return np.column_stack([x, y])

poly = Polygon()
for r in city['rings']:                           # even-odd, as the plates fill them (enclaves are holes)
    poly = poly.symmetric_difference(Polygon(to_miles(r)).buffer(0))
x0, y0, x1, y1 = poly.bounds
xs = np.arange(np.floor(x0 / STEP) * STEP - 2 * STEP, x1 + 3 * STEP, STEP)
ys = np.arange(np.floor(y0 / STEP) * STEP - 2 * STEP, y1 + 3 * STEP, STEP)
GX, GY = np.meshgrid(xs, ys)
INSIDE = contains_xy(poly, GX, GY)
NEAR = contains_xy(poly.buffer(0.3), GX, GY)      # computed a little past the limits, for smoothing
nodes = np.column_stack([GX[NEAR], GY[NEAR]])
log.info('lattice %d x %d, %d points inside the city', len(ys), len(xs), INSIDE.sum())

# the two starts and the hospital, in the same miles
rp = np.array([c for f in rle['features'] for c in (f['geometry']['coordinates'] if f['geometry']['type'] == 'LineString'
               else [q for part in f['geometry']['coordinates'] for q in part])])
south130 = -1 - (13000 - 1200) / 800                            # 130th St, 13000 S
rx, ry = miles_from_origin(rp[:, 1], rp[:, 0])
k130 = np.argmin(np.abs(ry - south130))
START = np.array([rx[k130], ry[k130]])
LOOP = np.array([0.0, 0.0])
HOSP = np.array(miles_from_origin(*HOSPITAL), float)
log.info('130th St on the alignment: %.3f, %.3f mi (%.5f, %.5f)', *START, rp[k130][1], rp[k130][0])

## 5. Analyze
Earliest arrival by connection scan (Dibbelt et al. 2013) from each start, leaving every 5 minutes; the median of the 12 runs is the travel time.

In [ ]:
NT = int(conn[:, 4].max()) + 1
CONN = conn.tolist()
DEP = conn[:, 0]


def scan(start, origin):
    best = [np.inf] * len(pts)
    for s in tree.query_ball_point(origin, ACCESS):
        best[s] = start + np.hypot(*(pts[s] - origin)) / WALK
    reached = bytearray(NT)
    end = start + HORIZON
    for dep, arr, a, b, t in CONN[int(np.searchsorted(DEP, start)):]:
        if dep > end:
            break
        if reached[t] or best[a] <= dep:
            reached[t] = 1
            if arr < best[b]:
                best[b] = arr
                for n, w in fp[b]:
                    if arr + w < best[n]:
                        best[n] = arr + w
    return np.array(best)


def minutes(origin, targets, departs=DEPARTS):
    """Median door-to-door minutes from origin to each target over the departures."""
    near = tree.query_ball_point(targets, ACCESS)
    direct = np.hypot(*(targets - origin).T) / WALK
    runs = []
    for start in departs:
        best = scan(start, origin)
        t = direct.copy()
        for i, ss in enumerate(near):
            if ss:
                ss = np.array(ss)
                t[i] = min(t[i], (best[ss] - start + np.hypot(*(pts[ss] - targets[i]).T) / WALK).min())
        runs.append(t / 60)
    return np.median(runs, axis=0)


targets = np.vstack([nodes, START, LOOP, HOSP])
T = {}
for name, origin in (('start', START), ('loop', LOOP)):
    m = minutes(origin, targets)
    T[name] = {'nodes': m[:-3], 'start': m[-3], 'loop': m[-2], 'hosp': m[-1]}
    log.info('%s: to State & Madison %.1f, to 130th %.1f, to the hospital %.1f min', name, m[-2], m[-3], m[-1])

In [ ]:
def grid_of(v):
    G = np.full(GX.shape, np.nan)
    G[NEAR] = np.where(v > WATER, np.nan, v)
    return G


def smooth(G, sigma, mask):
    ok = ~np.isnan(G)
    S = gaussian_filter(np.where(ok, G, 0), sigma) / np.maximum(gaussian_filter(ok.astype(float), sigma), 1e-6)
    S[~mask | ~ok & ~INSIDE] = np.nan
    return S


RAW = {n: grid_of(T[n]['nodes']) for n in T}
water = INSIDE & np.isnan(RAW['start'])
log.info('points with no stop within reach (water): %d', water.sum())

# residents: each 2020 block at its internal point takes the minutes of the nearest lattice point
bx, by = miles_from_origin(blocks['lat'], blocks['lon'])
ok = ~np.isnan(RAW['start']) & ~np.isnan(RAW['loop'])
kd = cKDTree(np.column_stack([GX[ok], GY[ok]]))
d, j = kd.query(np.column_stack([bx, by]))
far = d > 0.25
log.info('blocks over a quarter mile from a lattice point: %d (%d residents), max %.2f mi', far.sum(), blocks['pop'][far].sum(), d.max())
print(blocks[far][['geoid', 'pop', 'lat', 'lon']].head(8))
pop = blocks['pop'].to_numpy()
res = {n: RAW[n][ok][j] for n in RAW}


def share_within(v, mins, w=pop):
    return float((w * (v <= mins)).sum() / w.sum())


def wmedian(v, w=pop):
    o = np.argsort(v)
    c = np.cumsum(w[o])
    return float(v[o][np.searchsorted(c, c[-1] / 2)])


area = {n: RAW[n][INSIDE & ok] for n in RAW}
summary = {
    'residents': int(pop.sum()),
    'within60': {n: share_within(res[n], 60) for n in res},
    'within30': {n: share_within(res[n], 30) for n in res},
    'within90': {n: share_within(res[n], 90) for n in res},
    'median': {n: wmedian(res[n]) for n in res},
    'area60': {n: float((area[n] <= 60).mean()) for n in area},
    'startToLoop': float(T['start']['loop']), 'startToHosp': float(T['start']['hosp']),
    'loopToStart': float(T['loop']['start']), 'loopToHosp': float(T['loop']['hosp']),
    'startMiles': float(np.hypot(*START)), 'hospMiles': float(np.hypot(*(HOSP - START))),
}
summary

## 6. QA
Another hour (7-8 a.m.) should tell the same story; tooltips and findings use unsmoothed minutes.

In [ ]:
qa = {}
for name, origin in (('start', START), ('loop', LOOP)):
    m = minutes(origin, targets, departs=list(range(7 * 3600, 8 * 3600, 300)))
    qa[name] = share_within(grid_of(m[:-3])[ok][j], 60)
    log.info('%s, 7-8 a.m.: %.0f%% of residents within an hour (8-9 a.m.: %.0f%%); hospital %.0f min',
             name, 100 * qa[name], 100 * summary['within60'][name], m[-1])
    assert abs(qa[name] - summary['within60'][name]) < 0.1, 'the hour changes the story'
# the start's own neighbourhood is reachable on foot; the Loop is the Loop
assert RAW['loop'][np.argmin(np.abs(ys)), np.argmin(np.abs(xs))] < 6
assert summary['startToLoop'] > 3 * summary['loopToHosp']

## 7. Export
Minutes rounded to 0.1 for drawing; the hover squares (every half mile) carry unsmoothed minutes.

In [ ]:
CLS = [30, 45, 60, 75, 90, 120]                 # classes: under 30, 30-45, 45-60 | 60-75, 75-90, 90-120, 120+ (split at the hour)
cls = lambda v: int(np.searchsorted(CLS, v, side='right'))
A = smooth(RAW['start'], 0.8, NEAR)             # main map: light smoothing (0.2 mi) against speckle
S = {n: smooth(RAW[n], 1.5, INSIDE) for n in RAW}   # the cartograms: 0.4 mi

rows = lambda G, f: [[None if np.isnan(v) else f(v) for v in r] for r in G]
half = (np.isclose(GX % 0.5, 0) | np.isclose(GX % 0.5, 0.5)) & (np.isclose(GY % 0.5, 0) | np.isclose(GY % 0.5, 0.5))
cells = [[round(float(x), 2), round(float(y), 2), round(float(a)), round(float(b))]
         for x, y, a, b, h, i in zip(GX.ravel(), GY.ravel(), RAW['start'].ravel(), RAW['loop'].ravel(), half.ravel(), INSIDE.ravel())
         if h and i and not np.isnan(a) and not np.isnan(b)]
plate = dict(
    feeds={k: dict(name=FEEDS[k][1], snapshot=p.stem.split('_')[-1], url=FEEDS[k][2], trips=len(trips[k])) for k, p in feeds.items()},
    serviceDay=SERVICE_DAY.isoformat(), departs=['8:00', '8:55'],
    method=dict(walkMps=WALK_MPS, detour=DETOUR, accessMi=ACCESS, xferMi=XFER, xferS=XFER_S, step=STEP),
    lattice=dict(x0=round(float(xs[0]), 2), y0=round(float(ys[0]), 2), step=STEP, nx=len(xs), ny=len(ys)),
    classes=CLS,
    mapClass=[''.join('.' if np.isnan(v) else str(cls(v)) for v in r) for r in A],
    start=rows(S['start'], lambda v: round(float(v), 1)),
    loop=rows(S['loop'], lambda v: round(float(v), 1)),
    cells=cells,
    points=dict(start=START.round(3).tolist(), hosp=HOSP.round(3).tolist(), loop=[0, 0]),
    summary=summary, qa=qa,
    sources=dict(rleFaq='https://www.transitchicago.com/rle/faq/', osm='https://www.openstreetmap.org/copyright',
                 census='https://www2.census.gov/programs-surveys/decennial/2020/data/01-Redistricting_File--PL_94-171/Illinois/il2020.pl.zip'),
)
p130, pl = (round(100 * summary['within60'][n]) for n in ('start', 'loop'))
finding = f'From 130th Street, {p130}% of Chicagoans live within an hour by bus and train. From the Loop, {pl}% do.'
ledger = Ledger(
    source='CTA, Metra and Pace GTFS', dataset_id='GTFS schedules', url=FEEDS['cta'][2],
    pulled=date.fromisoformat(feeds['cta'].stem.split('_')[-1]), units='door-to-door minutes',
    method=['earliest arrival by connection scan, leaving every 5 minutes 8:00-8:55 a.m.; median of 12 runs',
            'walking: straight line x 1.3 at 1.2 m/s'],
    caveats=['scheduled service, not real-time', 'fares differ between CTA, Metra and Pace'],
)
write_plate(OUT / 'plate.json', day=15, ledger=ledger, finding=finding, payload=plate)
print(finding)
print('plate.json', round((OUT / 'plate.json').stat().st_size / 1024), 'KB')